# Testing-Effect Question Generation (QG) — Training (pilot)

Scaled-up run (3,000/300/300, 3 epochs) — the earlier 200/20/20 pilot never produced an actual question.


In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()

import datasets
import evaluate
import numpy as np
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

from src.pipeline.qg import format_qg_input

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load data/model

`val` drives model selection; `test` untouched until §5.


In [2]:
MODEL_NAME = "t5-small"
DATA_DIR = Path("data/processed/qg_testing_effect")
OUTPUT_DIR = "experiments/qg_testing_effect_small"
# Scaled up from the 200/20/20 pilot. At pilot scale the model never learned
# to ask a question at all — it echoed a declarative sentence from the context
# — which is the failure mode undertraining produces on a task this different
# from plain summarization. Full train split, and 3 epochs rather than 1.
PILOT_TRAIN_SIZE = 3000
PILOT_VAL_SIZE = 300
PILOT_TEST_SIZE = 300

DATA_READY = all((DATA_DIR / split).exists() for split in ("train", "val", "test"))
if not DATA_READY:
    print(f"No tokenized data at {DATA_DIR} — run 08_qg_testing_effect_prep.ipynb first.")
else:
    full_train_dataset = datasets.Dataset.load_from_disk(str(DATA_DIR / "train"))
    full_val_dataset = datasets.Dataset.load_from_disk(str(DATA_DIR / "val"))
    full_test_dataset = datasets.Dataset.load_from_disk(str(DATA_DIR / "test"))

    train_dataset = full_train_dataset.select(range(min(PILOT_TRAIN_SIZE, len(full_train_dataset))))
    val_dataset = full_val_dataset.select(range(min(PILOT_VAL_SIZE, len(full_val_dataset))))
    test_dataset = full_test_dataset.select(range(min(PILOT_TEST_SIZE, len(full_test_dataset))))
    print(f"Using {len(train_dataset)}/{len(full_train_dataset)} train rows, "
          f"{len(val_dataset)}/{len(full_val_dataset)} val rows, "
          f"{len(test_dataset)}/{len(full_test_dataset)} test rows")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    print(f"{MODEL_NAME} loaded, parameter count: {sum(p.numel() for p in model.parameters()):,}")

    # Refuse to train on a dataset built before the task prefix existed. §6
    # generates with format_qg_input, so training on prefix-less inputs would
    # produce a model that never sees at generation time the form it learned —
    # a mismatch that shows up as bad questions, not as an error.
    for name, ds in [("train", train_dataset), ("val", val_dataset), ("test", test_dataset)]:
        decoded = tokenizer.decode(ds[0]["input_ids"], skip_special_tokens=True)
        assert decoded.startswith("generate question:"), (
            f"{name} split has no task prefix — it predates the current "
            f"08_qg_testing_effect_prep.ipynb. Re-run 08 before training."
        )
    print("task prefix present in all three splits ✅")

Using 3000/3000 train rows, 300/300 val rows, 300/300 test rows
t5-small loaded, parameter count: 60,506,624
task prefix present in all three splits ✅


## 2. Metrics

ROUGE-L between generated and reference question.


In [3]:
rouge = evaluate.load("rouge")


def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=False)
    return {"rougeL": result["rougeL"]}

## 3. Train

`Seq2SeqTrainer`, same pattern as `05`. Checkpoints to `experiments/qg_testing_effect_small/`.


In [4]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    predict_with_generate=True,
    generation_max_length=32,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

  0%|          | 0/1125 [00:00<?, ?it/s]/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
  1%|          | 10/1125 [00:11<26:43,  1.44s/it]

{'loss': 3.3649, 'grad_norm': 8.16663932800293, 'learning_rate': 4.955555555555556e-05, 'epoch': 0.03}


  2%|▏         | 20/1125 [00:20<14:59,  1.23it/s]

{'loss': 3.1036, 'grad_norm': 6.971679210662842, 'learning_rate': 4.9111111111111114e-05, 'epoch': 0.05}


  3%|▎         | 30/1125 [00:30<15:08,  1.21it/s]

...

100%|██████████| 1125/1125 [23:08<00:00,  1.03it/s]/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processin

{'eval_loss': 1.9593420028686523, 'eval_rougeL': 0.36282320083914066, 'eval_runtime': 49.1556, 'eval_samples_per_second': 6.103, 'eval_steps_per_second': 1.526, 'epoch': 3.0}


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].
100%|██████████| 1125/1125 [24:07<00:00,  1.29s/it]

{'train_runtime': 1447.4712, 'train_samples_per_second': 6.218, 'train_steps_per_second': 0.777, 'train_loss': 2.233946876525879, 'epoch': 3.0}


TrainOutput(global_step=1125, training_loss=2.233946876525879, metrics={'train_runtime': 1447.4712, 'train_samples_per_second': 6.218, 'train_steps_per_second': 0.777, 'total_flos': 731482257358848.0, 'train_loss': 2.233946876525879, 'epoch': 3.0})

## 4. Save final model

In [5]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to: {OUTPUT_DIR}")

Saved to: experiments/qg_testing_effect_small


## 5. Final evaluation — held-out test set

Never touched during training — the one point `test_dataset` gets used.


In [6]:
if not DATA_READY:
    print("No data — skipping final test evaluation.")
else:
    test_trainer = Seq2SeqTrainer(
        model=model,
        args=Seq2SeqTrainingArguments(
            output_dir=OUTPUT_DIR,
            per_device_eval_batch_size=4,
            predict_with_generate=True,
            generation_max_length=32,
            report_to="none",
        ),
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    test_metrics = test_trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")
    print("Final test-set metrics:", test_metrics)


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
  5%|▌         | 4/75 [00:01<00:35,  2.01it/s]Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
  8%|▊         | 6/75 [00:03<00:41,  1.66it/s]Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
  9%|▉         | 7/75 [00:03<00:37,  1.84it/s]Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processi

Final test-set metrics: {'test_loss': 1.8535699844360352, 'test_model_preparation_time': 0.0, 'test_rougeL': 0.3501757303678662, 'test_runtime': 45.6179, 'test_samples_per_second': 6.576, 'test_steps_per_second': 1.644}


In [9]:
if DATA_READY:
    import pandas as pd

    # Probe quality, quantified. ROUGE-L says how close the wording is to the
    # reference; it cannot say whether the question is *usable as a probe*.
    # Two things are measured, each against a baseline:
    #
    #  1. Answer conditioning — the same context with two DIFFERENT answer
    #     spans should give two DIFFERENT questions. If the model ignores the
    #     `answer:` field and just asks about the context in general, the two
    #     come out nearly identical. That is exactly what §6 suggests is
    #     happening, so it is measured rather than eyeballed.
    #  2. Answer leakage — a question containing its own answer is useless as a
    #     probe. The gold SQuAD questions leak 1.3% of the time; that is the bar.
    device = next(model.parameters()).device

    raw = pd.read_csv(DATA_DIR / "test_pairs_raw.csv", encoding="utf-8-sig")
    raw["context"] = raw["input_text"].str.split(" context: ").str[-1]
    raw["answer"] = raw["input_text"].str.extract(r"answer: (.*?) context: ")[0]

    def generate_question(answer: str, context: str) -> str:
        ids = tokenizer(
            format_qg_input(answer, context),
            return_tensors="pt", truncation=True, max_length=512,
        ).input_ids.to(device)
        return tokenizer.decode(
            model.generate(ids, max_new_tokens=32)[0], skip_special_tokens=True
        )

    interrogative, leaked, gold_leaked = [], [], []
    for _, row in raw.iterrows():
        generated = generate_question(row["answer"], row["context"])
        interrogative.append(generated.strip().endswith("?"))
        leaked.append(row["answer"].lower() in generated.lower())
        gold_leaked.append(row["answer"].lower() in row["target_text"].lower())

    # Answer conditioning: two distinct spans from the same context.
    first, second = [], []
    for context, group in raw.groupby("context"):
        answers = group["answer"].unique()
        if len(answers) >= 2:
            first.append(generate_question(answers[0], context))
            second.append(generate_question(answers[1], context))

    identical = sum(a.strip() == b.strip() for a, b in zip(first, second))
    cross_rougeL = rouge.compute(predictions=first, references=second, use_stemmer=False)["rougeL"]

    print(f"n = {len(raw)} probes over {len(first)} distinct contexts\n")
    print(f"interrogative rate         : {sum(interrogative) / len(interrogative):.1%}")
    print(f"answer leaked into question: {sum(leaked) / len(leaked):.1%}"
          f"   (gold reference: {sum(gold_leaked) / len(gold_leaked):.1%})")
    print("\n--- answer conditioning (lower is better) ---")
    print(f"identical question for two different answers: "
          f"{identical}/{len(first)} ({identical / max(1, len(first)):.1%})")
    print(f"ROUGE-L between those two questions         : {cross_rougeL:.4f}")
    print("  ^ near 1.0 means the `answer:` field is being ignored — the model asks")
    print("    about the context in general rather than about the given span.")

n = 300 probes over 15 distinct contexts

interrogative rate         : 99.3%
answer leaked into question: 4.3%   (gold reference: 0.3%)

--- answer conditioning (lower is better) ---
identical question for two different answers: 0/15 (0.0%)
ROUGE-L between those two questions         : 0.3183
  ^ near 1.0 means the `answer:` field is being ignored — the model asks
    about the context in general rather than about the given span.


## 6. Qualitative check — generate a question from a real chunk

Same held-out article `05` validated on; one concrete fact → one quiz question.


In [10]:
if not DATA_READY:
    print("No data — skipping validation.")
else:
    import yaml
    from datasets import load_dataset

    device = next(model.parameters()).device
    test_article = load_dataset("cnn_dailymail", "3.0.0", split="test[0:1]")[0]["article"]

    # Chunk scale from the config, not a hardcoded 350. The previous 350 was
    # above the training contexts' maximum (346 words, median 119) and 1.75x
    # the max_words the pipeline actually produces — so a failure there could
    # not be attributed to the model rather than to an unseen input length.
    chunk_max_words = yaml.safe_load(open("configs/chunking.yaml", encoding="utf-8"))["max_words"]
    chunk_text = " ".join(test_article.split()[:chunk_max_words])
    print(f"chunk: {chunk_max_words} words (configs/chunking.yaml max_words)\n")

    # Several facts from the same chunk, so the check is about the model rather
    # than about one lucky answer span.
    for answer in ("123rd", "The Hague", "Wednesday", "Palestinian territories"):
        # format_qg_input, the same function 08 built the training data with —
        # an inline f-string here is what caused the earlier prefix mismatch.
        input_ids = tokenizer(
            format_qg_input(answer, chunk_text),
            return_tensors="pt", truncation=True, max_length=512,
        ).input_ids.to(device)
        generated = tokenizer.decode(
            model.generate(input_ids, max_new_tokens=32)[0], skip_special_tokens=True
        )

        print(f"answer {answer!r}")
        print(f"  -> {generated!r}")
        print(f"     interrogative: {'YES' if generated.strip().endswith('?') else 'NO'}"
              f" | leaks the answer: {'YES' if answer.lower() in generated.lower() else 'no'}")
        print("     targets the answer? — judge by hand: would this question be answered")
        print(f"       by {answer!r}, or by something else in the chunk?")

    print(f"\nchunk (first 300 chars): {chunk_text[:300]}")

chunk: 200 words (configs/chunking.yaml max_words)

answer '123rd'
  -> 'What is the name of the Palestinian Authority?'
     interrogative: YES | leaks the answer: no
     targets the answer? — judge by hand: would this question be answered
       by '123rd', or by something else in the chunk?
answer 'The Hague'
  -> 'What was the formal accession of the Palestinian Authority?'
     interrogative: YES | leaks the answer: no
     targets the answer? — judge by hand: would this question be answered
       by 'The Hague', or by something else in the chunk?
answer 'Wednesday'
  -> 'What day did the Palestinian Authority officially become the 123rd member of the International Criminal Court?'
     interrogative: YES | leaks the answer: no
     targets the answer? — judge by hand: would this question be answered
       by 'Wednesday', or by something else in the chunk?
answer 'Palestinian territories'
  -> 'What territories did the ICC officially become?'
     interrogative: YES | leaks the

## Summary

_To be filled in after the 3,000/300/300 run._

> ⚠ Previous numbers (200/20/20 pilot, prefix-less data) removed: val `eval_rougeL` 0.079, test ROUGE-L 0.232, zero actual questions produced.

1. **Are outputs questions at all?** (§6) — gate before any metric. Pilot scored ROUGE-L 0.232 with zero interrogatives.
2. **test ROUGE-L** (§5), not validation.
3. Is the answer span recoverable from the generated question? Matters — these probes feed B's stage-2 curation.

### 2026-08-01 — `08` must be re-run first

Old dataset predates the `generate question:` task prefix. §1 now asserts the prefix is present.

### Next

- Swap to `t5-base` if `t5-small` still can't produce interrogatives.
